In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import optuna
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
import pygeohash as pgh

def time_to_minutes(t):
    if pd.isnull(t): return np.nan
    h, m = map(int, str(t).split(':'))
    return h * 60 + m

# ==========================================
# UPGRADE 4: Optuna Tuning Function
# ==========================================
def tune_lgbm_with_optuna(X_train, y_train, X_val, y_val):
    print("\n--- Running Optuna Hyperparameter Tuning for LightGBM ---")

    lgb_train = lgb.Dataset(X_train, y_train)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'verbosity': -1,
            'boosting_type': 'gbdt',
            'device_type': 'gpu', # Change to 'cpu' if no GPU available for LightGBM

            'feature_pre_filter': False,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 8),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.9),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.9),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        }

        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_val],
            num_boost_round=10000,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        preds = model.predict(X_val)
        return np.sqrt(mean_squared_error(y_val, preds))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print("\nBest LightGBM Parameters found:")
    print(study.best_params)
    return study.best_params


def create_safe_features(train_path, test_path):
    print("1. Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df['is_test'] = 0
    test_df['is_test'] = 1

    combined = pd.concat([train_df, test_df], ignore_index=True)
    combined['hour'] = combined['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else np.nan)
    combined['minutes'] = combined['timestamp'].apply(time_to_minutes)

    print("2. Engineering Base Features...")
    combined['lat'] = combined['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else np.nan)
    combined['lon'] = combined['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else np.nan)
    combined['time_sin'] = np.sin(2 * np.pi * combined['minutes'] / 1440)
    combined['time_cos'] = np.cos(2 * np.pi * combined['minutes'] / 1440)

    combined['LargeVehicles'] = combined['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(-1)
    combined['Landmarks'] = combined['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(-1)
    combined['Temperature'] = combined['Temperature'].fillna(combined['Temperature'].median())
    combined['NumberofLanes'] = combined['NumberofLanes'].fillna(combined['NumberofLanes'].median())

    # ==========================================
    # UPGRADE 2: Categorical Cross-Features
    # ==========================================
    print("3. Engineering Categorical Cross-Features...")
    combined['Weather_Road'] = combined['Weather'].astype(str) + "_on_" + combined['RoadType'].astype(str)

    print("4. K-Means Spatial Clustering...")
    coords = combined[['lat', 'lon']].dropna()
    kmeans = KMeans(n_clusters=30, random_state=42, n_init=10)
    kmeans.fit(coords)
    combined.loc[coords.index, 'spatial_cluster'] = kmeans.labels_
    combined['spatial_cluster'] = combined['spatial_cluster'].fillna(-1).astype(int)

    print("5. Calculating Safe Historical Demand (Train-Only)...")
    train_only = combined[combined['is_test'] == 0]

    geo_means = train_only.groupby('geohash')['demand'].mean().rename('hist_geo_mean')
    geo_hour_means = train_only.groupby(['geohash', 'hour'])['demand'].mean().rename('hist_geo_hour_mean')
    cluster_means = train_only.groupby('spatial_cluster')['demand'].mean().rename('hist_cluster_mean')

    combined = combined.merge(geo_means, on='geohash', how='left')
    combined = combined.merge(geo_hour_means, on=['geohash', 'hour'], how='left')
    combined = combined.merge(cluster_means, on='spatial_cluster', how='left')

    global_mean = train_only['demand'].mean()
    combined['hist_geo_mean'] = combined['hist_geo_mean'].fillna(global_mean)
    combined['hist_geo_hour_mean'] = combined['hist_geo_hour_mean'].fillna(combined['hist_geo_mean'])
    combined['hist_cluster_mean'] = combined['hist_cluster_mean'].fillna(global_mean)

    print("6. Processing Categoricals...")
    # Included the new Weather_Road feature
    cat_features = ['geohash', 'RoadType', 'Weather', 'Weather_Road']
    for col in cat_features:
        combined[col] = combined[col].fillna('Missing').astype(str)

    train_full = combined[combined['is_test'] == 0].drop(columns=['is_test', 'Index', 'timestamp', 'minutes', 'hour'])
    test_full = combined[combined['is_test'] == 1].drop(columns=['is_test', 'Index', 'timestamp', 'minutes', 'hour', 'demand'])

    xgb_train = train_full.copy()
    xgb_test = test_full.copy()
    for col in cat_features:
        le = LabelEncoder()
        le.fit(combined[col])
        xgb_train[col] = le.transform(xgb_train[col])
        xgb_test[col] = le.transform(xgb_test[col])

    return train_full, test_full, xgb_train, xgb_test, cat_features

if __name__ == "__main__":
    cb_train, cb_test, xgb_train, xgb_test, cat_cols = create_safe_features('dataset/train.csv', 'dataset/test.csv')

    max_day = cb_train['day'].max()
    y_full = cb_train['demand']

    train_mask = cb_train['day'] < max_day
    val_mask = cb_train['day'] == max_day
    y_tr, y_va = y_full[train_mask], y_full[val_mask]

    X_cb_tr = cb_train[train_mask].drop(columns=['day', 'demand'])
    X_cb_va = cb_train[val_mask].drop(columns=['day', 'demand'])
    X_cb_full = cb_train.drop(columns=['day', 'demand'])
    X_cb_test = cb_test.drop(columns=['day'])

    X_xgb_tr = xgb_train[train_mask].drop(columns=['day', 'demand'])
    X_xgb_va = xgb_train[val_mask].drop(columns=['day', 'demand'])
    X_xgb_full = xgb_train.drop(columns=['day', 'demand'])
    X_xgb_test = xgb_test.drop(columns=['day'])

    # ---------------------------------------------------------
    # OPTIONAL: Run Optuna Tuning
    # Uncomment the two lines below to find new optimal LightGBM parameters
    # ---------------------------------------------------------
    best_lgbm_params = tune_lgbm_with_optuna(X_xgb_tr, y_tr, X_xgb_va, y_va)
    print(best_lgbm_params)

    # ---------------------------------------------------------
    # 1. TRAIN LIGHTGBM
    # ---------------------------------------------------------
    print("\n--- Training LightGBM Model ---")
    lgb_train = lgb.Dataset(X_xgb_tr, y_tr)
    lgb_val = lgb.Dataset(X_xgb_va, y_va, reference=lgb_train)

    lgb_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.015,
        'max_depth': 6,
        'num_leaves': 31,
        'feature_fraction': 0.8,
        'seed': 42,
        'verbosity': -1
    }

    lgb_model = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=10000,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(stopping_rounds=1500), lgb.log_evaluation(500)]
    )

    print("Retraining LightGBM on 100% data...")
    lgb_train_full = lgb.Dataset(X_xgb_full, y_full)
    lgb_final = lgb.train(lgb_params, lgb_train_full, num_boost_round=lgb_model.best_iteration)
    lgb_preds = lgb_final.predict(X_xgb_test)

    # ---------------------------------------------------------
    # 2. TRAIN CATBOOST
    # ---------------------------------------------------------
    print("\n--- Training CatBoost Model ---")
    cb_model = CatBoostRegressor(
        iterations=20000, learning_rate=0.015, depth=6,
        task_type='GPU', eval_metric='RMSE', random_seed=42
    )
    cb_model.fit(X_cb_tr, y_tr, eval_set=(X_cb_va, y_va), cat_features=cat_cols, early_stopping_rounds=1500, verbose=500)

    print("Retraining CatBoost on 100% data...")
    cb_final = CatBoostRegressor(
        iterations=cb_model.get_best_iteration(), learning_rate=0.015, depth=6,
        task_type='GPU', random_seed=42
    )
    cb_final.fit(X_cb_full, y_full, cat_features=cat_cols, verbose=False)
    cb_preds = cb_final.predict(X_cb_test)

    # ---------------------------------------------------------
    # 3. TRAIN XGBOOST
    # ---------------------------------------------------------
    print("\n--- Training XGBoost Model ---")
    dtrain = xgb.DMatrix(X_xgb_tr, label=y_tr)
    dval = xgb.DMatrix(X_xgb_va, label=y_va)

    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'device': 'cuda',
        'learning_rate': 0.015,
        'max_depth': 6,
        'min_child_weight': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.7
    }
    xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=10000, evals=[(dval, 'Validation')], early_stopping_rounds=1500, verbose_eval=500)

    print("Retraining XGBoost on 100% data...")
    dtrain_full = xgb.DMatrix(X_xgb_full, label=y_full)
    dtest = xgb.DMatrix(X_xgb_test)
    xgb_final = xgb.train(xgb_params, dtrain_full, num_boost_round=xgb_model.best_iteration)
    xgb_preds = xgb_final.predict(dtest)

    # ==========================================
    # UPGRADE 5: The 3-Way Blend
    # ==========================================
    print("\n--- Finalizing 3-Way Blend Predictions ---")

    # Clip all predictions to prevent mathematically impossible negative demand
    cb_preds = np.clip(cb_preds, 0, None)
    xgb_preds = np.clip(xgb_preds, 0, None)
    lgb_preds = np.clip(lgb_preds, 0, None)

    # The ultimate 3-way blend. CatBoost and XGBoost take the lead, LightGBM acts as the tie-breaker/smoother.
    final_predictions = (cb_preds * 0.4) + (xgb_preds * 0.4) + (lgb_preds * 0.2)

    test_raw = pd.read_csv('dataset/test.csv')
    sub = pd.DataFrame({
        'Index': test_raw['Index'],
        'demand': final_predictions
    })

    sub.to_csv('submission_3way_blend11.csv', index=False)
    print("Done! Saved 'submission_3way_blend11.csv'")

1. Loading datasets...
2. Engineering Base Features...
3. Engineering Categorical Cross-Features...
4. K-Means Spatial Clustering...


[I 2026-06-04 19:07:25,822] A new study created in memory with name: no-name-bbed5356-b79b-4dd9-ad96-dc641179d41b


5. Calculating Safe Historical Demand (Train-Only)...
6. Processing Categoricals...

--- Running Optuna Hyperparameter Tuning for LightGBM ---


[I 2026-06-04 19:07:27,848] Trial 0 finished with value: 0.05772655095982304 and parameters: {'learning_rate': 0.006385274089869063, 'max_depth': 4, 'num_leaves': 63, 'feature_fraction': 0.7708581990049685, 'bagging_fraction': 0.6217330081254182, 'bagging_freq': 3, 'min_child_samples': 34}. Best is trial 0 with value: 0.05772655095982304.
[I 2026-06-04 19:07:31,234] Trial 1 finished with value: 0.06044501312221642 and parameters: {'learning_rate': 0.007011917843414725, 'max_depth': 6, 'num_leaves': 31, 'feature_fraction': 0.6618912196395347, 'bagging_fraction': 0.655759130776172, 'bagging_freq': 2, 'min_child_samples': 35}. Best is trial 0 with value: 0.05772655095982304.
[I 2026-06-04 19:07:33,141] Trial 2 finished with value: 0.056183694281540665 and parameters: {'learning_rate': 0.0067194943739474575, 'max_depth': 4, 'num_leaves': 28, 'feature_fraction': 0.5598495309407787, 'bagging_fraction': 0.6389769995129115, 'bagging_freq': 4, 'min_child_samples': 10}. Best is trial 2 with valu


Best LightGBM Parameters found:
{'learning_rate': 0.010185786147051321, 'max_depth': 4, 'num_leaves': 15, 'feature_fraction': 0.5003181441682393, 'bagging_fraction': 0.7678379041131842, 'bagging_freq': 4, 'min_child_samples': 10}
{'learning_rate': 0.010185786147051321, 'max_depth': 4, 'num_leaves': 15, 'feature_fraction': 0.5003181441682393, 'bagging_fraction': 0.7678379041131842, 'bagging_freq': 4, 'min_child_samples': 10}

--- Training LightGBM Model ---
Training until validation scores don't improve for 1500 rounds
[500]	valid_0's rmse: 0.0644965
[1000]	valid_0's rmse: 0.0662061
[1500]	valid_0's rmse: 0.0672276
Early stopping, best iteration is:
[176]	valid_0's rmse: 0.061998
Retraining LightGBM on 100% data...

--- Training CatBoost Model ---
0:	learn: 0.1398512	test: 0.1437452	best: 0.1437452 (0)	total: 22.5ms	remaining: 7m 30s
500:	learn: 0.0230795	test: 0.0625895	best: 0.0594267 (204)	total: 8.95s	remaining: 5m 48s
1000:	learn: 0.0222372	test: 0.0643336	best: 0.0594267 (204)	to